# 统计基线模型：SARIMA & Prophet

思路：先从最简单的Naive baseline开始，然后逐步加复杂度。如果简单模型就能打到不错的效果，说明数据的pattern比较规律，没必要上太复杂的模型。

**实验设置：**
- 训练集: 2015.1 - 2017.12 (36 months)
- 测试集: 2018.1 - 2018.12 (12 months)
- 评估指标: RMSE, MAE, MAPE

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.arima_model import SarimaForecaster, naive_forecast, seasonal_naive
from src.models.prophet_model import ProphetForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, save_figure

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)

print(f'Train: {len(train)} months ({train.index[0].strftime("%Y-%m")} ~ {train.index[-1].strftime("%Y-%m")})')
print(f'Test:  {len(test)} months ({test.index[0].strftime("%Y-%m")} ~ {test.index[-1].strftime("%Y-%m")})')

## 1. Naive Baselines

两个最简单的baseline：
- **Naive**: 把最后一个月的值一直重复（假设未来=现在）
- **Seasonal Naive**: 用去年同月的值（假设今年=去年）

这两个虽然简单，但在很多场景下比你想象的要强。

In [ ]:
naive_pred = naive_forecast(train, test)
snaive_pred = seasonal_naive(train, test)

results = []
results.append(evaluate_forecast(test, naive_pred, 'Naive'))
results.append(evaluate_forecast(test, snaive_pred, 'Seasonal Naive'))

for r in results:
    print(f"{r['model']:20s}  RMSE={r['RMSE']:>10.2f}  MAE={r['MAE']:>10.2f}  MAPE={r['MAPE']:>6.2f}%")

In [ ]:
fig = plot_forecast(train, test, snaive_pred, 'Seasonal Naive Forecast')
plt.show()

Seasonal Naive的RMSE大约是Naive的一半，说明季节性确实是主要的pattern。MAPE在25%左右，对月度销售来说算是可以接受的baseline了。

接下来看SARIMA能不能在这个基础上再改进。

## 2. SARIMA

用`pmdarima`的`auto_arima`来搜索最优参数。从EDA我们知道 d=1, m=12，D=1（季节差分），剩下的p/q让它自己找。

In [ ]:
sarima = SarimaForecaster()
sarima.auto_fit(train, seasonal=True, m=12)

In [ ]:
sarima_pred = sarima.predict(steps=len(test))
sarima_pred.index = test.index

results.append(evaluate_forecast(test, sarima_pred, 'SARIMA'))
print(f"SARIMA: RMSE={results[-1]['RMSE']:.2f}, MAE={results[-1]['MAE']:.2f}, MAPE={results[-1]['MAPE']:.2f}%")

In [ ]:
fig = plot_forecast(train, test, sarima_pred, 'SARIMA Forecast')
save_figure(fig, 'sarima_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
# 诊断一下残差
print(sarima.get_summary())

SARIMA的RMSE比Seasonal Naive高一些...有点出乎意料。看预测曲线的话，SARIMA在趋势方向上是对的，但某些月份偏差比较大。可能是因为数据量太少，参数估计不够稳定。

不过MAPE这个指标比较受极端值影响，从MAE来看两个模型差距没那么大。

## 3. Prophet

Prophet的好处是自动处理holiday effect和changepoint detection。我们加上US holidays看看有没有帮助。

In [ ]:
prophet = ProphetForecaster(
    yearly_seasonality=True,
    changepoint_prior_scale=0.05  # default, controls trend flexibility
)
prophet.fit(train)

In [ ]:
prophet_pred = prophet.predict(periods=len(test))
prophet_pred.index = test.index

results.append(evaluate_forecast(test, prophet_pred.values, 'Prophet'))
print(f"Prophet: RMSE={results[-1]['RMSE']:.2f}, MAE={results[-1]['MAE']:.2f}, MAPE={results[-1]['MAPE']:.2f}%")

In [ ]:
fig = plot_forecast(train, test, prophet_pred.values, 'Prophet Forecast')
save_figure(fig, 'prophet_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
# 看看Prophet拆解出来的components
forecast_df = prophet.get_components(periods=len(test))
fig = prophet.model.plot_components(forecast_df)
plt.tight_layout()
plt.show()

Prophet的表现中规中矩。从components图能看出它检测到的trend和seasonality都是合理的。

可以试试调高`changepoint_prior_scale`让trend更灵活，但数据量这么小，怕过拟合。先这样。

## 4. 模型对比

In [ ]:
comparison = compare_models(results)
print(comparison.to_string())

In [ ]:
# 所有模型放在一张图上
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train.values, label='Train', color='gray', alpha=0.5)
ax.plot(test.index, test.values, label='Actual', color='black', linewidth=2)
ax.plot(test.index, sarima_pred.values, label='SARIMA', linestyle='--')
ax.plot(test.index, prophet_pred.values, label='Prophet', linestyle='--')
ax.plot(test.index, snaive_pred.values, label='Seasonal Naive', linestyle=':')
ax.legend()
ax.set_title('All Models vs Actual (2018)')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 小结

有意思的是Seasonal Naive反而是RMSE最低的。这说明在这个数据上，"去年同月"本身就是一个很强的信号，复杂模型并没有从仅有的36个训练点里学到更多有用的东西。

当然这也跟test set只有一年有关，如果做rolling forecast可能结论会不同。

下一步在 `03_deep_learning.ipynb` 里试LSTM，不抱太大期望但还是想看看。